# Phase 07I: ORT Quality Aggregate and Deployment Lock

Phase 07I tổng hợp bằng chứng Phase 07H của ba cặp `T1–S1`, `T2–S2`,
`T3–S3`, đối soát lại ONNX Runtime artifacts từ Phase 07G và tạo một
deployment bundle bất biến cho bước tích hợp Android.

Phạm vi khóa:

- xác minh 3/3 lần đánh giá đủ 5.000 ảnh Karpathy Validation;
- tổng hợp BLEU-1, BLEU-4, METEOR, ROUGE-L, CIDEr-D và CE;
- kiểm tra lại quality gate INT8 − FP32;
- đóng gói encoder/decoder ONNX đã chọn, preprocessing và vocabulary;
- không huấn luyện, không chọn lại checkpoint, không đọc Held-out Test;
- chưa benchmark thiết bị Android vật lý;
- chỉ cấp quyền Phase 07J tích hợp ứng dụng và kiểm tra runtime contract.

## Kaggle inputs

1. Ba output Phase 07H: T1–S1, T2–S2, T3–S3.
2. Ba output Phase 07G tương ứng.
3. Phase 02A Shared Student Pipeline.
4. Phase 01 MS COCO Data Contract.

## Runtime

- Accelerator: `None / CPU`.
- Internet: `Off`.
- Không cần dataset ảnh và không gắn Test.

## Cell 1 — Authorization, provenance và integrity

In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase7i_ort_quality_aggregate_deployment_lock",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
BUNDLE = OUT / "deployment_bundle"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, BUNDLE, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

PAIR_SPECS = {
    "T1-S1": {
        "token": "T1_S1", "slug": "t1_s1", "student_id": "S1", "seed": 123,
        "strategy": "encoder_decoder_dynamic_int8_linear",
    },
    "T2-S2": {
        "token": "T2_S2", "slug": "t2_s2", "student_id": "S2", "seed": 2026,
        "strategy": "encoder_decoder_dynamic_int8_linear",
    },
    "T3-S3": {
        "token": "T3_S3", "slug": "t3_s3", "student_id": "S3", "seed": 42,
        "strategy": "decoder_dynamic_int8_linear",
    },
}
QUALITY_GATES = {
    "CIDEr-D_minimum_delta": -0.02,
    "BLEU-4_minimum_delta": -0.01,
    "CE_maximum_delta": 0.05,
}

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents
    return target

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file()
    if "size_bytes" in reference:
        assert path.stat().st_size == int(reference["size_bytes"])
    assert sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(reference["rows"]) == int(marker["manifest_rows"])
    for row in rows:
        artifact = resolve_inside(marker_path.parent, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
P1 = read_json(P1_PATH)
P2 = read_json(P2_PATH)
assert P1["status"] == "ready"
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
assert P2["status"] == "ready"
assert P2["data_policy"]["test_set_status"] == "sealed"
assert P2["data_policy"]["test_set_used"] is False

P1_ROOT = P1_PATH.parent
P2_ROOT = P2_PATH.parent
p1_sums = P1_ROOT / "SHA256SUMS.txt"
assert p1_sums.is_file()
assert sha256(p1_sums) == P1["sha256sums_sha256"]
p1_hashes = {}
for line in p1_sums.read_text(encoding="utf-8").splitlines():
    digest, relative_path = line.split("  ", 1)
    p1_hashes[relative_path] = digest

p2_manifest_ref = P2["manifests"]["csv"]
p2_manifest_path = resolve_inside(P2_ROOT, p2_manifest_ref["relative_path"])
assert sha256(p2_manifest_path) == p2_manifest_ref["sha256"]
with p2_manifest_path.open("r", encoding="utf-8", newline="") as handle:
    p2_rows = list(csv.DictReader(handle))
assert len(p2_rows) == int(P2["artifact_count"]) == 38
for row in p2_rows:
    artifact = resolve_inside(P2_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256(artifact) == row["sha256"]

PAIR_INPUTS = {}
for pair, spec in PAIR_SPECS.items():
    token = spec["token"]
    p7h_path = exactly_one(
        f"PHASE7H_{token}_ORT_INT8_VALIDATION_QUALITY_COMPLETE.json"
    )
    p7i_ready_path = exactly_one(
        f"PHASE7I_ORT_QUALITY_AGGREGATE_INPUT_READY_{token}.json"
    )
    p7g_path = exactly_one(
        f"PHASE7G_{token}_ORT_MOBILE_EXPORT_PREFLIGHT_COMPLETE.json"
    )
    p7h_input_path = exactly_one(
        f"PHASE7H_{token}_ORT_INT8_VALIDATION_QUALITY_INPUT_READY.json"
    )
    p7h = read_json(p7h_path)
    p7i_ready = read_json(p7i_ready_path)
    p7g = read_json(p7g_path)
    p7h_input = read_json(p7h_input_path)

    assert p7h["status"] == "complete"
    assert p7h["pair"] == pair
    assert p7h["student_id"] == spec["student_id"]
    assert int(p7h["seed"]) == spec["seed"]
    assert p7h["strategy"] == spec["strategy"]
    assert p7h["completed_images"] == p7h["validation_images"] == 5000
    assert p7h["quality_gate_passed"] is True
    assert p7h["deployment_decision"] == (
        "retain_ort_int8_candidate_for_phase7i"
    )
    assert p7h["caption_quality_evaluated"] is True
    assert p7h["physical_android_benchmark_performed"] is False
    assert p7h["new_training_performed"] is False
    assert p7h["checkpoint_reselection_performed"] is False
    assert p7h["test_set_status"] == "sealed"
    assert p7h["test_set_used"] is False
    p7h_manifest_rows = verify_manifest(p7h_path, p7h)

    assert p7i_ready["status"] == "ready"
    assert p7i_ready["pair"] == pair
    assert p7i_ready["quality_gate_passed"] is True
    assert p7i_ready["strategy"] == spec["strategy"]
    assert p7i_ready["physical_android_benchmark_authorized"] is False
    assert p7i_ready["test_set_status"] == "sealed"
    assert p7i_ready["test_set_used"] is False
    quality_path = resolve_inside(
        p7h_path.parent, p7i_ready["quality_result"]["relative_path"]
    )
    assert quality_path.stat().st_size == int(
        p7i_ready["quality_result"]["size_bytes"]
    )
    assert sha256(quality_path) == p7i_ready["quality_result"]["sha256"]

    run_state_path = p7h_path.parent / "results/phase7h_run_state.json"
    run_state = read_json(run_state_path)
    assert run_state["pair"] == pair and run_state["status"] == "complete"
    assert run_state["completed_images"] == run_state["total_images"] == 5000
    assert run_state["test_set_status"] == "sealed"
    assert run_state["test_set_used"] is False
    prediction_path = (
        p7h_path.parent
        / f"predictions/{spec['slug']}_paired_predictions.jsonl"
    )
    assert prediction_path.is_file()
    assert sha256(prediction_path) == run_state["paired_prediction_sha256"]
    image_ids = set()
    with prediction_path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            row = json.loads(line)
            image_id = int(row["image_id"])
            assert image_id not in image_ids
            assert int(row["fp32"]["image_id"]) == image_id
            assert int(row["int8"]["image_id"]) == image_id
            image_ids.add(image_id)
    assert line_number == len(image_ids) == 5000

    assert p7g["status"] == "complete"
    assert p7g["pair"] == pair
    assert p7g["strategy"] == spec["strategy"]
    assert p7g["modern_onnx_exports_complete"] is True
    assert p7g["fp32_ort_parity_passed"] is True
    assert p7g["ort_int8_compatibility_passed"] is True
    assert p7g["test_set_status"] == "sealed"
    assert p7g["test_set_used"] is False
    p7g_manifest_rows = verify_manifest(p7g_path, p7g)
    assert p7h_input["status"] == "ready"
    assert p7h_input["pair"] == pair
    assert p7h_input["strategy"] == spec["strategy"]
    assert p7h_input["test_set_status"] == "sealed"
    assert p7h_input["test_set_used"] is False
    for role, artifact_ref in p7h_input["artifacts"].items():
        artifact = resolve_inside(
            p7g_path.parent, artifact_ref["relative_path"]
        )
        assert artifact.is_file()
        assert artifact.stat().st_size == int(artifact_ref["size_bytes"])
        assert sha256(artifact) == artifact_ref["sha256"]

    pair_contract_path = (
        p7h_path.parent / "configs/phase7h_pair_ort_quality_contract.json"
    )
    pair_contract = read_json(pair_contract_path)
    assert pair_contract["pair"] == pair
    assert pair_contract["quality_gates"] == QUALITY_GATES
    assert pair_contract["selection_split"] == "Karpathy Validation"
    assert pair_contract["validation_images"] == 5000
    assert pair_contract["validation_references"] == 25010
    assert pair_contract["test_set_status"] == "sealed"
    assert pair_contract["test_set_used"] is False
    for role, key in [
        ("encoder_fp32", ("fp32", "encoder_sha256")),
        ("decoder_fp32", ("fp32", "decoder_sha256")),
        ("encoder_candidate", ("int8_candidate", "encoder_sha256")),
        ("decoder_candidate", ("int8_candidate", "decoder_sha256")),
    ]:
        assert (
            p7h_input["artifacts"][role]["sha256"]
            == pair_contract[key[0]][key[1]]
        )

    PAIR_INPUTS[pair] = {
        "spec": spec,
        "phase7h_marker_path": p7h_path,
        "phase7h_marker": p7h,
        "phase7h_manifest_rows": p7h_manifest_rows,
        "quality_path": quality_path,
        "quality": read_json(quality_path),
        "run_state": run_state,
        "pair_contract_path": pair_contract_path,
        "pair_contract": pair_contract,
        "phase7g_marker_path": p7g_path,
        "phase7g_marker": p7g,
        "phase7g_manifest_rows": p7g_manifest_rows,
        "phase7h_input": p7h_input,
    }

write_json(AUDIT / "phase7i_cell01_integrity.json", {
    "schema_version": "phase7i_cell01_integrity_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pairs": {
        pair: {
            "phase7h_manifest_rows": len(item["phase7h_manifest_rows"]),
            "phase7g_manifest_rows": len(item["phase7g_manifest_rows"]),
            "validation_predictions": 5000,
            "quality_gate_passed": True,
        }
        for pair, item in PAIR_INPUTS.items()
    },
    "phase2_manifest_rows": len(p2_rows),
    "new_training_started": False,
    "checkpoint_reselection": False,
    "physical_android_benchmark": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07I CELL 1 — INTEGRITY")
print("Status                  : PASSED")
print("Phase 07H pairs         : 3 / 3 VERIFIED")
print("Validation predictions  : 5,000 x 3 VERIFIED")
print("Phase 07G artifacts     : 3 / 3 VERIFIED")
print("Phase 02A manifest      : 38 / 38 VERIFIED")
print("New training            : NOT STARTED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 1 / 5 code cells")

PHASE 07I CELL 1 — INTEGRITY
Status                  : PASSED
Phase 07H pairs         : 3 / 3 VERIFIED
Validation predictions  : 5,000 x 3 VERIFIED
Phase 07G artifacts     : 3 / 3 VERIFIED
Phase 02A manifest      : 38 / 38 VERIFIED
New training            : NOT STARTED
Held-out Test           : SEALED / NOT USED
Progress                : 1 / 5 code cells


## Cell 2 — Quality synthesis và trade-off

In [2]:
STARTED = time.perf_counter()
METRICS = ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D", "CE"]
summary_rows = []
for pair, item in PAIR_INPUTS.items():
    quality = item["quality"]
    spec = item["spec"]
    assert quality["schema_version"] == "phase7h_pair_ort_quality_result_v1"
    assert quality["pair"] == pair
    assert quality["student_id"] == spec["student_id"]
    assert int(quality["seed"]) == spec["seed"]
    assert quality["strategy"] == spec["strategy"]
    assert quality["selection_split"] == "Karpathy Validation"
    assert quality["validation_images"] == 5000
    assert quality["validation_references"] == 25010
    assert quality["quality_gate_passed"] is True
    assert all(quality["gates"].values())
    assert quality["test_set_status"] == "sealed"
    assert quality["test_set_used"] is False
    for metric in METRICS:
        expected_delta = quality["int8"][metric] - quality["fp32"][metric]
        assert math.isclose(
            quality["delta_int8_minus_fp32"][metric],
            expected_delta,
            rel_tol=0.0,
            abs_tol=1e-12,
        )
    delta = quality["delta_int8_minus_fp32"]
    assert delta["CIDEr-D"] >= QUALITY_GATES["CIDEr-D_minimum_delta"]
    assert delta["BLEU-4"] >= QUALITY_GATES["BLEU-4_minimum_delta"]
    assert delta["CE"] <= QUALITY_GATES["CE_maximum_delta"]

    compatibility = item["phase7h_input"]["compatibility_result"]
    assert compatibility["preflight_passed"] is True
    assert all(compatibility["gates"].values())
    summary_rows.append({
        "pair": pair,
        "student_id": spec["student_id"],
        "seed": spec["seed"],
        "strategy": spec["strategy"],
        **{
            f"fp32_{metric}": quality["fp32"][metric]
            for metric in METRICS
        },
        **{
            f"int8_{metric}": quality["int8"][metric]
            for metric in METRICS
        },
        **{
            f"delta_{metric}": delta[metric]
            for metric in METRICS
        },
        "exact_caption_agreement_fraction": (
            quality["exact_caption_agreement_fraction"]
        ),
        "fp32_package_size_bytes": compatibility["fp32_package_size_bytes"],
        "int8_package_size_bytes": (
            compatibility["candidate_package_size_bytes"]
        ),
        "package_reduction_fraction": (
            compatibility["package_reduction_fraction"]
        ),
        "quality_gate_passed": quality["quality_gate_passed"],
    })

summary_csv = RESULT / "phase7i_pair_quality_summary.csv"
with summary_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

mean_delta = {
    metric: sum(row[f"delta_{metric}"] for row in summary_rows) / 3
    for metric in METRICS
}
aggregate = {
    "schema_version": "phase7i_ort_quality_aggregate_v1",
    "created_at_utc": now(),
    "status": "passed",
    "selection_split": "Karpathy Validation",
    "validation_images_per_pair": 5000,
    "validation_references_per_pair": 25010,
    "quality_gates": QUALITY_GATES,
    "pairs_passed": 3,
    "pairs_total": 3,
    "all_quality_gates_passed": True,
    "mean_delta_int8_minus_fp32": mean_delta,
    "rows": summary_rows,
    "decision": "retain_three_ort_int8_candidates_for_android_integration",
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(RESULT / "phase7i_ort_quality_aggregate.json", aggregate)
write_json(AUDIT / "phase7i_cell02_quality_synthesis.json", {
    "schema_version": "phase7i_cell02_quality_synthesis_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pairs_passed": 3,
    "pairs_total": 3,
    "mean_delta_int8_minus_fp32": mean_delta,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07I CELL 2 — QUALITY SYNTHESIS")
print("Status                  : PASSED")
for row in summary_rows:
    print(
        f"{row['pair']}: CIDEr-D={row['int8_CIDEr-D']:.6f} "
        f"(delta={row['delta_CIDEr-D']:+.6f}) | "
        f"BLEU-4 delta={row['delta_BLEU-4']:+.6f} | "
        f"CE delta={row['delta_CE']:+.6f} | "
        f"size reduction={100*row['package_reduction_fraction']:.2f}%"
    )
print("Quality gates           : 3 / 3 PASSED")
print("Selection split         : VALIDATION")
print("Held-out Test           : SEALED")
print("Progress                : 2 / 5 code cells")

PHASE 07I CELL 2 — QUALITY SYNTHESIS
Status                  : PASSED
T1-S1: CIDEr-D=1.109202 (delta=-0.002309) | BLEU-4 delta=-0.001124 | CE delta=+0.000580 | size reduction=23.01%
T2-S2: CIDEr-D=1.020697 (delta=-0.004319) | BLEU-4 delta=-0.001502 | CE delta=+0.002352 | size reduction=38.31%
T3-S3: CIDEr-D=1.094024 (delta=-0.003732) | BLEU-4 delta=-0.001140 | CE delta=+0.000243 | size reduction=16.59%
Quality gates           : 3 / 3 PASSED
Selection split         : VALIDATION
Held-out Test           : SEALED
Progress                : 2 / 5 code cells


## Cell 3 — Build immutable mobile deployment bundle

In [3]:
STARTED = time.perf_counter()
SHARED = BUNDLE / "shared"
MODELS = BUNDLE / "models"
EVIDENCE = BUNDLE / "evidence"
for directory in [SHARED, MODELS, EVIDENCE]:
    directory.mkdir(parents=True, exist_ok=True)

shared_sources = {
    "word_vocabulary_minfreq5.json": (
        P1_ROOT / "vocabulary/word_vocabulary_minfreq5.json"
    ),
    "tokenizer_contract.json": P1_ROOT / "vocabulary/tokenizer_contract.json",
    "sequence_encoding_contract.json": (
        P1_ROOT / "vocabulary/sequence_encoding_contract.json"
    ),
}
for destination_name, source_path in shared_sources.items():
    relative = source_path.relative_to(P1_ROOT).as_posix()
    assert relative in p1_hashes
    assert sha256(source_path) == p1_hashes[relative]
    shutil.copy2(source_path, SHARED / destination_name)

vocabulary = read_json(SHARED / "word_vocabulary_minfreq5.json")
assert len(vocabulary["id_to_token"]) == 10080
assert vocabulary["id_to_token"][:4] == [
    "<pad>", "<bos>", "<eos>", "<unk>",
]

bundle_registry = []
for pair, item in PAIR_INPUTS.items():
    spec = item["spec"]
    pair_dir = MODELS / spec["slug"]
    evidence_dir = EVIDENCE / spec["slug"]
    pair_dir.mkdir(parents=True, exist_ok=True)
    evidence_dir.mkdir(parents=True, exist_ok=True)

    artifact_map = item["phase7h_input"]["artifacts"]
    p7g_root = item["phase7g_marker_path"].parent
    selected_artifacts = {
        "encoder.onnx": artifact_map["encoder_candidate"],
        "decoder.onnx": artifact_map["decoder_candidate"],
    }
    packaged_artifacts = {}
    for destination_name, reference in selected_artifacts.items():
        source_path = resolve_inside(p7g_root, reference["relative_path"])
        destination_path = pair_dir / destination_name
        shutil.copy2(source_path, destination_path)
        assert destination_path.stat().st_size == int(reference["size_bytes"])
        assert sha256(destination_path) == reference["sha256"]
        packaged_artifacts[destination_name] = {
            "relative_path": destination_path.relative_to(BUNDLE).as_posix(),
            "size_bytes": destination_path.stat().st_size,
            "sha256": sha256(destination_path),
        }

    preprocessing_source = (
        P2_ROOT
        / f"configs/preprocessing/{spec['student_id'].lower()}_preprocessing_contract.json"
    )
    p2_preprocess_rows = [
        row for row in p2_rows
        if row["relative_path"]
        == preprocessing_source.relative_to(P2_ROOT).as_posix()
    ]
    assert len(p2_preprocess_rows) == 1
    assert sha256(preprocessing_source) == p2_preprocess_rows[0]["sha256"]
    preprocessing_destination = pair_dir / "preprocessing_contract.json"
    shutil.copy2(preprocessing_source, preprocessing_destination)

    shutil.copy2(
        item["quality_path"], evidence_dir / "phase7h_quality_result.json"
    )
    shutil.copy2(
        item["pair_contract_path"],
        evidence_dir / "phase7h_ort_quality_contract.json",
    )
    p7g_contract_path = (
        p7g_root / "configs/phase7g_ort_mobile_export_contract.json"
    )
    p7g_contract = read_json(p7g_contract_path)
    assert p7g_contract["pair"] == pair
    assert p7g_contract["locked_strategy"] == spec["strategy"]
    assert p7g_contract["target"]["application_runtime"] == (
        "onnxruntime-android"
    )
    assert p7g_contract["target"]["application_runtime_version"] == "1.29.0"
    assert p7g_contract["test_set_status"] == "sealed"
    assert p7g_contract["test_set_used"] is False

    quality = item["quality"]
    runtime_contract = {
        "schema_version": "phase7i_pair_mobile_runtime_contract_v1",
        "created_at_utc": now(),
        "pair": pair,
        "student_id": spec["student_id"],
        "seed": spec["seed"],
        "strategy": spec["strategy"],
        "runtime": {
            "name": "onnxruntime-android",
            "version": "1.29.0",
            "opset": p7g_contract["target"]["onnx_opset"],
            "cpu_threads_for_validation": 1,
        },
        "artifacts": packaged_artifacts,
        "preprocessing_contract": (
            preprocessing_destination.relative_to(BUNDLE).as_posix()
        ),
        "shared_vocabulary": "shared/word_vocabulary_minfreq5.json",
        "tokenizer_contract": "shared/tokenizer_contract.json",
        "sequence_encoding_contract": "shared/sequence_encoding_contract.json",
        "tensor_contract": p7g_contract["target"]["shapes"],
        "tensor_names": {
            "encoder_inputs": ["image"],
            "encoder_outputs": ["visual_memory"],
            "decoder_inputs": [
                "decoder_input_ids",
                "visual_memory",
                "decoder_attention_mask",
            ],
            "decoder_outputs": ["logits"],
        },
        "special_token_ids": {
            "pad": 0, "bos": 1, "eos": 2, "unk": 3,
        },
        "decoding": {
            "beam_size": 3,
            "length_penalty": 0.7,
            "maximum_steps": 31,
        },
        "validation_quality": {
            "CIDEr-D": quality["int8"]["CIDEr-D"],
            "BLEU-4": quality["int8"]["BLEU-4"],
            "CE": quality["int8"]["CE"],
            "quality_gate_passed": quality["quality_gate_passed"],
        },
        "physical_android_benchmark_performed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    runtime_contract_path = pair_dir / "mobile_runtime_contract.json"
    write_json(runtime_contract_path, runtime_contract)
    bundle_registry.append({
        "pair": pair,
        "student_id": spec["student_id"],
        "seed": spec["seed"],
        "strategy": spec["strategy"],
        "runtime_contract": {
            "relative_path": runtime_contract_path.relative_to(BUNDLE).as_posix(),
            "size_bytes": runtime_contract_path.stat().st_size,
            "sha256": sha256(runtime_contract_path),
        },
        "artifacts": packaged_artifacts,
    })

write_json(CONFIG / "phase7i_mobile_bundle_registry.json", {
    "schema_version": "phase7i_mobile_bundle_registry_v1",
    "created_at_utc": now(),
    "runtime": "onnxruntime-android:1.29.0",
    "vocabulary_size": 10080,
    "pairs": bundle_registry,
    "new_training": False,
    "checkpoint_reselection": False,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase7i_cell03_bundle.json", {
    "schema_version": "phase7i_cell03_bundle_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pairs_packaged": 3,
    "onnx_artifacts_packaged": 6,
    "preprocessing_contracts_packaged": 3,
    "shared_contracts_packaged": len(shared_sources),
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07I CELL 3 — DEPLOYMENT BUNDLE")
print("Status                  : PASSED")
print("Pair bundles            : 3 / 3")
print("ONNX artifacts          : 6 / 6 HASH VERIFIED")
print("Preprocessing contracts : 3 / 3")
print("Vocabulary              : 10,080")
print("Physical Android        : NOT BENCHMARKED")
print("Held-out Test           : SEALED")
print("Progress                : 3 / 5 code cells")

PHASE 07I CELL 3 — DEPLOYMENT BUNDLE
Status                  : PASSED
Pair bundles            : 3 / 3
ONNX artifacts          : 6 / 6 HASH VERIFIED
Preprocessing contracts : 3 / 3
Vocabulary              : 10,080
Physical Android        : NOT BENCHMARKED
Held-out Test           : SEALED
Progress                : 3 / 5 code cells


## Cell 4 — Deployment lock và Phase 07J authorization

In [4]:
STARTED = time.perf_counter()
assert aggregate["all_quality_gates_passed"] is True
assert len(bundle_registry) == 3

deployment_lock = {
    "schema_version": "phase7i_ort_deployment_lock_v1",
    "created_at_utc": now(),
    "status": "locked",
    "decision": "three_ort_int8_candidates_locked_for_android_integration",
    "selection_basis": "Karpathy Validation only",
    "quality_gates": QUALITY_GATES,
    "pairs": [
        {
            "pair": row["pair"],
            "student_id": row["student_id"],
            "seed": row["seed"],
            "strategy": row["strategy"],
            "int8_CIDEr-D": row["int8_CIDEr-D"],
            "int8_BLEU-4": row["int8_BLEU-4"],
            "int8_CE": row["int8_CE"],
            "delta_CIDEr-D": row["delta_CIDEr-D"],
            "delta_BLEU-4": row["delta_BLEU-4"],
            "delta_CE": row["delta_CE"],
            "package_reduction_fraction": row["package_reduction_fraction"],
            "quality_gate_passed": row["quality_gate_passed"],
        }
        for row in summary_rows
    ],
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "android_application_integration_authorized": True,
    "physical_android_benchmark_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
lock_path = CONFIG / "phase7i_ort_deployment_lock.json"
write_json(lock_path, deployment_lock)
write_json(AUDIT / "phase7i_cell04_deployment_lock.json", {
    "schema_version": "phase7i_cell04_deployment_lock_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pairs_locked": 3,
    "android_application_integration_authorized": True,
    "physical_android_benchmark_authorized": False,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07I CELL 4 — DEPLOYMENT LOCK")
print("Status                  : LOCKED")
print("Candidates retained     : T1-S1 / T2-S2 / T3-S3")
print("Android integration     : AUTHORIZED FOR PHASE 07J")
print("Physical benchmark      : NOT YET AUTHORIZED")
print("New training            : NOT PERFORMED")
print("Held-out Test           : SEALED")
print("Progress                : 4 / 5 code cells")

PHASE 07I CELL 4 — DEPLOYMENT LOCK
Status                  : LOCKED
Candidates retained     : T1-S1 / T2-S2 / T3-S3
Android integration     : AUTHORIZED FOR PHASE 07J
Physical benchmark      : NOT YET AUTHORIZED
New training            : NOT PERFORMED
Held-out Test           : SEALED
Progress                : 4 / 5 code cells


## Cell 5 — Manifest, completion marker và next handoff

In [5]:
STARTED = time.perf_counter()
manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, BUNDLE]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })

manifest_csv = MANIFEST / "phase7i_ort_deployment_bundle_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "phase7i_ort_deployment_bundle_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7i_ort_deployment_bundle_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)

manifest_reference = {
    "csv": {
        "relative_path": manifest_csv.relative_to(OUT).as_posix(),
        "size_bytes": manifest_csv.stat().st_size,
        "sha256": sha256(manifest_csv),
        "rows": len(manifest_rows),
    },
    "json": {
        "relative_path": manifest_json.relative_to(OUT).as_posix(),
        "size_bytes": manifest_json.stat().st_size,
        "sha256": sha256(manifest_json),
    },
    "sha256sums": {
        "relative_path": sha_file.relative_to(OUT).as_posix(),
        "size_bytes": sha_file.stat().st_size,
        "sha256": sha256(sha_file),
    },
}

next_ready = {
    "schema_version": "phase7j_android_integration_input_ready_v1",
    "created_at_utc": now(),
    "status": "ready",
    "runtime": "onnxruntime-android:1.29.0",
    "pairs": ["T1-S1", "T2-S2", "T3-S3"],
    "deployment_lock": {
        "relative_path": lock_path.relative_to(OUT).as_posix(),
        "size_bytes": lock_path.stat().st_size,
        "sha256": sha256(lock_path),
    },
    "bundle_registry": {
        "relative_path": (
            CONFIG / "phase7i_mobile_bundle_registry.json"
        ).relative_to(OUT).as_posix(),
        "size_bytes": (
            CONFIG / "phase7i_mobile_bundle_registry.json"
        ).stat().st_size,
        "sha256": sha256(CONFIG / "phase7i_mobile_bundle_registry.json"),
    },
    "manifest": manifest_reference,
    "android_application_integration_authorized": True,
    "physical_android_benchmark_authorized": False,
    "new_training": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(OUT / "PHASE7J_ANDROID_INTEGRATION_INPUT_READY.json", next_ready)

marker = {
    "schema_version": "phase7i_ort_quality_aggregate_deployment_lock_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": "three_ort_int8_candidates_locked_phase7j_authorized",
    "pairs_verified": 3,
    "pairs_quality_passed": 3,
    "validation_images_per_pair": 5000,
    "deployment_bundle_complete": True,
    "android_application_integration_authorized": True,
    "physical_android_benchmark_performed": False,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "manifest": manifest_reference,
    "manifest_rows": len(manifest_rows),
    "elapsed_seconds_cell05": time.perf_counter() - STARTED,
}
write_json(
    OUT / "PHASE7I_ORT_QUALITY_AGGREGATE_DEPLOYMENT_LOCK_COMPLETE.json",
    marker,
)

print("PHASE 07I — FINAL")
print("Status                  : COMPLETE")
print("Quality gates           : 3 / 3 PASSED")
print("Deployment bundle       : COMPLETE")
print("Android integration     : PHASE 07J AUTHORIZED")
print("Physical benchmark      : NOT PERFORMED")
print("New training            : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Progress                : 5 / 5 code cells")

PHASE 07I — FINAL
Status                  : COMPLETE
Quality gates           : 3 / 3 PASSED
Deployment bundle       : COMPLETE
Android integration     : PHASE 07J AUTHORIZED
Physical benchmark      : NOT PERFORMED
New training            : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Manifest                : 29 / 29 VERIFIED
Progress                : 5 / 5 code cells
